**Local ARENA · chapter1 transformer interp · additional**

Use the **ARENA (local GPU)** kernel selected for this notebook. Run the local setup cell first. Package installation and Linux/Colab paths have been adapted; the original is retained in `arena/ARENA_3.0`. Exercise blanks are intentional. API-dependent sections require your own credentials and are run manually.

[Original notebook](https://github.com/callummcdougall/ARENA_3.0/blob/527f9376b40ad9a12ecd80490884b0009b54dd55/chapter1_transformer_interp/exercises/monthly_algorithmic_problems/july23_palindromes/training_model.ipynb)

In [ ]:
from pathlib import Path
import runpy
import sys
workspace = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "arena" / "runtime.py").is_file())
local_arena = runpy.run_path(str(workspace / "arena" / "runtime.py"))
globals().update(local_arena["setup"]('chapter1_transformer_interp', 'monthly_algorithmic_problems/july23_palindromes', 'training_model.ipynb'))

## Setup

In [ ]:
import json
import os
import sys
import webbrowser
from functools import partial
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple, Union

import circuitsvis as cv
import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from jaxtyping import Bool, Float, Int
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from torch import Tensor
from tqdm import tqdm
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig, utils
from transformer_lens.components import LayerNorm
from transformer_lens.hook_points import HookPoint

# Make sure exercises are in the path
chapter = r"chapter1_transformers"
exercises_dir = Path(f"{os.getcwd().split(chapter)[0]}/{chapter}/exercises").resolve()
section_dir = exercises_dir / "monthly_algorithmic_problems" / "june23_palindromes"
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

from monthly_algorithmic_problems.july23_palindromes.model import create_model
from monthly_algorithmic_problems.july23_palindromes.training import TrainArgs, train

device = t.device("cuda" if t.cuda.is_available() else "cpu")

MAIN = __name__ == "__main__"

## Transformer

In [ ]:
# Took about 10 minutes to train

args = TrainArgs(
    half_length=10,
    max_value=30,
    trainset_size=100_000,
    valset_size=5_000,
    epochs=15,
    batch_size=512,
    lr=1e-3,
    weight_decay=0.0,
    seed=42,
    d_model=28,
    d_head=14,
    n_heads=2,
    d_mlp=None,
    normalization_type="LN",
    use_wandb=True,
    device=device,
)
model = train(args)

In [ ]:
# Took about 10 minutes to train

args = TrainArgs(
    half_length=10,
    max_value=30,
    trainset_size=100_000,
    valset_size=5_000,
    epochs=15,
    batch_size=512,
    lr=1e-3,
    weight_decay=0.0,
    seed=42,
    d_model=28,
    d_head=14,
    n_heads=2,
    d_mlp=None,
    normalization_type="LN",
    use_wandb=True,
    device=device,
)
model = train(args)

In [ ]:
# Save the model
filename = section_dir / "palindrome_classifier.pt"
t.save(model.state_dict(), filename)

# Check we can load in the model
model_loaded = create_model(
    half_length=10, max_value=30, seed=42, d_model=28, d_head=14, n_heads=2, normalization_type="LN", d_mlp=None
)
model_loaded.load_state_dict(t.load(filename))